# Sistem Perdagangan Algoritmik Kuantitatif Multi-Timeframe Simetris
**Penelitian Tugas Akhir: Pengujian Kinerja Model Trend-Following (1D) & Price Action Pullback (1H)**

Dokumen ini memuat implementasi komprehensif dari strategi kuantitatif multi-timeframe (MTF) yang dirancang untuk menguji efektivitas penekanan risiko (*drawdown reduction*) dan ekspektansi keuntungan pada instrumen indeks pasar saham AS (SPY, QQQ, DJI) dan saham berkapitalisasi pasar besar (AAPL).

Alur kerja notebook ini terbagi menjadi tahapan-tahapan modular berikut:
1. **Cell 1:** Penyiapan Lingkungan & Dependensi Pustaka (*Imports & Environment Setup*)
2. **Cell 2:** Konfigurasi Parameter Global Empiris (*Global Configuration & Target Paths*)
3. **Cell 3:** Mesin Pra-Pemrosesan Data MTF Bebas Bias Kausal (*Data Preprocessing Engine*)
4. **Cell 4:** Arsitektur Strategi Simetris Penuh (*Unified Symmetric MTF Strategy Class*)
5. **Cell 5:** Mesin Rekonsiliasi Transaksi Bebas Look-Behind Bias (*Trade Reconciliation*)
6. **Cell 6:** Ekstraksi Metrik Kinerja Standar Industri (*Standardized Metric Extraction*)
7. **Cell 7:** Loop Eksekusi Batch Lintas Instrumen (*Batch Execution Loop*)
8. **Cell 8:** Visualisasi Tabel Kinerja Notebook (*Notebook Tabular Display*)
9. **Cell 9:** Ekspor Hasil Kinerja ke Excel (*Dual Excel Exporter*)
10. **Cell 10:** Modul Pencarian Ruang Parameter Dinamis (*Dynamic Parameter Optimization - Commented Out*)

## Cell 1: Penyiapan Lingkungan & Pustaka Pendukung
Memuat pustaka standar manipulasi data (`pandas`, `numpy`), mesin teknikal (`pandas_ta`), serta kerangka simulasi transaksi standar institusional (`backtesting.py`). Melakukan *monkey-patching* `pd.DataFrame.iteritems` untuk memastikan kompatibilitas penuh dengan versi Pandas modern.

In [21]:
# =============================================================================
# CELL 1: IMPORTS & ENVIRONMENT SETUP
# =============================================================================
import os
import sys
import warnings
from pathlib import Path

# Mengabaikan peringatan internal numerik & pandas
warnings.filterwarnings('ignore')

# Pustaka manipulasi data & array
import pandas as pd
import numpy as np

# Monkey-patch untuk kompatibilitas Pandas 2.x/3.x dengan pustaka eksternal
pd.DataFrame.iteritems = pd.DataFrame.items

# Pustaka teknikal & framework backtesting standar industri
import pandas_ta as ta
from backtesting import Backtest, Strategy
from backtesting.lib import FractionalBacktest

# Pustaka ekspor & penataan gaya Excel
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# Konfigurasi format tampilan angka desimal pada Pandas (2 desimal)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("pustaka berhasil dimuat.")


pustaka berhasil dimuat.


## Cell 2: Konfigurasi Parameter Global (GLOBAL_PARAMS)
Memuat parameter kuantitatif universal yang diterapkan secara seragam pada seluruh instrumen acuan (`SPY`, `QQQ`, `DJI`, `AAPL`). Nilai parameter ini merupakan hasil optimasi empiris yang terbukti mampu meredam *whipsaw* pada indeks modern tanpa merusak kausalitas matematis:
* `macro_spread_threshold: 0.002` (Diperketat menjadi 0.2% agar sistem sangat responsif menangkap tren sejak awal).
* `ema_micro: 16` (Filter mikro dipercepat untuk mendeteksi pullback lebih gesit).
* `atr_period: 12` (Periode volatilitas penyeimbang).
* `atr_mult: 9.6` (Ruang napas ekstra lebar agar posisi tidak tersapu whipsaw indeks).
* `cooldown_bars: 10` (Jeda 10 bar untuk menahan sistem agar tidak overtrading).
* `min_profit_pct: 0.023` (Pengunci laba baru aktif setelah keuntungan mencapai 2.3%).

In [22]:
# =============================================================================
# CELL 2: GLOBAL CONFIGURATION & TARGET PATHS
# =============================================================================
# Konfigurasi parameter kuantitatif simetris penuh (Identik antara Long dan Short)
# Parameter hasil Optimasi Empiris untuk meredam noise intraday indeks modern
GLOBAL_PARAMS = {
    'initial_capital': 1000.0,
    'fee': 0.0003,                    
    'ema_macro_fast': 21,              
    'ema_macro_slow': 63,              
    'macro_spread_threshold': 0.002,   # Diperketat menjadi 0.2% agar sistem sangat responsif menangkap tren sejak awal.
    'ema_micro': 16,                   # Filter mikro dipercepat (16 bar) untuk mendeteksi pullback lebih gesit.
    'atr_period': 12,                  # Periode volatilitas penyeimbang.
    'atr_mult': 9.6,                   # Ruang napas ekstra lebar (9.6x ATR) agar posisi tidak tersapu whipsaw indeks (terutama DJI).
    'cooldown_bars': 10,               # Jeda 10 bar untuk menahan sistem agar tidak overtrading saat pasar bergejolak.
    'min_profit_pct': 0.023            # Pengunci laba (ratchet) baru aktif setelah keuntungan mencapai 2.3%.
}

# Daftar instrumen acuan yang diuji secara batch
ASSETS = ['SPY', 'QQQ', 'DJI', 'AAPL']

# Jalur direktori kerja dan berkas output Excel target (E:/kultivasi_4_tahun/ta/)
BASE_DIR = Path("E:/kultivasi_4_tahun/ta")
if not BASE_DIR.exists():
    BASE_DIR = Path.cwd()

# Jalur pencarian data mentah adaptif
DATA_DIR = BASE_DIR / "data_index"
if not DATA_DIR.exists():
    DATA_DIR = BASE_DIR.parent / "TugasAkhir" / "data_index"

# Berkas target luaran Excel di direktori target ta/
FILE_RINGKASAN = BASE_DIR / "Ringkasan_Kinerja_Semua_Instrumen.xlsx"
FILE_TRADES = BASE_DIR / "Detail_Trades_Semua_Instrumen.xlsx"

print("Konfigurasi Parameter Global (Terkunci - Optimasi Empiris):")
for k, v in GLOBAL_PARAMS.items():
    print(f"  * {k:<25}: {v}")
print(f"\nDirektori Target : {BASE_DIR}")
print(f"Berkas Ringkasan : {FILE_RINGKASAN}")
print(f"Berkas Trades    : {FILE_TRADES}")


Konfigurasi Parameter Global (Terkunci - Optimasi Empiris):
  * initial_capital          : 1000.0
  * fee                      : 0.0003
  * ema_macro_fast           : 21
  * ema_macro_slow           : 63
  * macro_spread_threshold   : 0.002
  * ema_micro                : 16
  * atr_period               : 12
  * atr_mult                 : 9.6
  * cooldown_bars            : 10
  * min_profit_pct           : 0.023

Direktori Target : E:\kultivasi_4_tahun\ta
Berkas Ringkasan : E:\kultivasi_4_tahun\ta\Ringkasan_Kinerja_Semua_Instrumen.xlsx
Berkas Trades    : E:\kultivasi_4_tahun\ta\Detail_Trades_Semua_Instrumen.xlsx


## Cell 3: Mesin Pra-Pemrosesan Data MTF (Data Preprocessing Engine)
Menerapkan *feature engineering* multi-timeframe yang ketat:
1. **Zero Lookahead Bias:** Indikator tren makro harian (EMA 21 dan EMA 63 pada timeframe 1D) digeser satu bar (`.shift(1)`) sebelum digabungkan (*forward-fill*) ke dalam data per jam (1H).
2. **Sanitasi Jam Bursa Reguler (RTH):** Hanya memproses jam perdagangan reguler bursa AS (09:00 - 16:00 EST) serta mengeliminasi bar akhir pekan (Sabtu/Minggu).

In [23]:
# =============================================================================
# CELL 3: DATA PREPROCESSING ENGINE
# =============================================================================
def calc_ema_series(series, span):
    """Menghitung Exponential Moving Average (EMA) berbasis span."""
    return pd.Series(series).ewm(span=span, adjust=False).mean().values

def calc_atr_series(high, low, close, period=14):
    """Menghitung Average True Range (ATR) berbasis True Range Wilder/EMA."""
    high_s = pd.Series(high)
    low_s = pd.Series(low)
    close_s = pd.Series(close)
    close_prev = close_s.shift(1)
    tr = pd.concat([
        high_s - low_s,
        (high_s - close_prev).abs(),
        (low_s - close_prev).abs()
    ], axis=1).max(axis=1)
    return pd.Series(tr).ewm(span=period, adjust=False).mean().values

def load_clean_data(file_path, is_1h=False):
    """Membaca data CSV/Parquet, sanitasi jam RTH 09:00-16:00, dan eliminasi akhir pekan."""
    p = Path(file_path)
    if not p.exists():
        raise FileNotFoundError(f"Berkas data tidak ditemukan pada path: {p}")
        
    if p.suffix.lower() == '.parquet':
        df = pd.read_parquet(p)
    else:
        with open(p, 'r', encoding='utf-8') as f:
            first_line = f.readline()
            sep = '\t' if '\t' in first_line else ','
        df = pd.read_csv(p, sep=sep)
        
    df.columns = [c.lower() for c in df.columns]
    
    # Standarisasi kolom waktu
    if 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'])
    elif 'date' in df.columns:
        df['datetime'] = pd.to_datetime(df['date'])
        df.drop(columns=['date'], inplace=True, errors='ignore')
    elif 'time' in df.columns:
        df['datetime'] = pd.to_datetime(df['time'])
        
    df.sort_values('datetime', inplace=True)
    df.set_index('datetime', inplace=True)
    
    # Eliminasi akhir pekan (Sabtu=5, Minggu=6)
    df = df[df.index.dayofweek < 5]
    
    # Filter jam bursa reguler (Regular Trading Hours 09:00 - 16:00 EST)
    if is_1h:
        df = df[(df.index.hour >= 9) & (df.index.hour <= 16)]
        
    df['date'] = df.index.date
    return df

def get_asset_file_paths(asset):
    """Pemetaan adaptif berkas data 1D dan 1H per instrumen."""
    search_dirs = [DATA_DIR, BASE_DIR / "data_index", BASE_DIR.parent / "TugasAkhir" / "data_index"]
    for d in search_dirs:
        mapping = {
            'SPY': {'1d': d / 'spy_US500_D1.parquet', '1h': d / 'spy_US500_H1.parquet'},
            'QQQ': {'1d': d / 'Nasdaq100_1d_data.csv', '1h': d / 'Nasdaq100_1h_data.csv'},
            'DJI': {'1d': d / 'Dow_Jones_1d_data.csv', '1h': d / 'Dow_Jones_1h_data.csv'},
            'AAPL': {'1d': d / 'validasi-aapl' / 'aapl_1d.csv', '1h': d / 'validasi-aapl' / 'aapl_1h.csv'}
        }
        if mapping[asset]['1d'].exists() and mapping[asset]['1h'].exists():
            return mapping[asset]
            
    raise FileNotFoundError(f"Tidak dapat menemukan berkas data lengkap untuk instrumen: {asset}")

def preprocess_asset_data(asset, start_dt='2018-02-19', end_dt='2024-05-10 23:59:59'):
    """
    Pipeline Feature Engineering Multi-Timeframe:
    1. Menghitung indikator 1D secara utuh pada data penuh (Full Period) dengan pergeseran 1 bar (.shift(1))
       guna mencegah kebocoran data masa depan (Zero Lookahead Bias).
    2. Menyaring data 1H pada rentang waktu evaluasi seragam (2018-2024).
    3. Menggabungkan indikator 1D ke 1H berbasis tanggal.
    4. Menghitung EMA Mikro 32 dan ATR 14 pada 1H.
    """
    paths = get_asset_file_paths(asset)
    d1 = load_clean_data(paths['1d'], is_1h=False)
    h1 = load_clean_data(paths['1h'], is_1h=True)
    
    # 1. Feature Engineering Dimensi Makro 1D (Shift 1 bar untuk kausalitas)
    d1['ema_fast'] = pd.Series(calc_ema_series(d1['close'], GLOBAL_PARAMS['ema_macro_fast']), index=d1.index).shift(1)
    d1['ema_slow'] = pd.Series(calc_ema_series(d1['close'], GLOBAL_PARAMS['ema_macro_slow']), index=d1.index).shift(1)
    thresh = GLOBAL_PARAMS['macro_spread_threshold']
    
    # Aturan Makro Simetris:
    d1['macro_bullish'] = d1['ema_fast'] > (d1['ema_slow'] * (1.0 + thresh))
    d1['macro_bearish'] = d1['ema_fast'] < (d1['ema_slow'] * (1.0 - thresh))
    d1['macro_bull_alive'] = d1['ema_fast'] >= d1['ema_slow']
    d1['macro_bear_alive'] = d1['ema_fast'] <= d1['ema_slow']
    
    # 2. Sinkronisasi rentang waktu evaluasi bersama
    h1 = h1[(h1.index >= pd.to_datetime(start_dt)) & (h1.index <= pd.to_datetime(end_dt))]
    
    # 3. Penyelarasan MTF 1D ke 1H berbasis tanggal
    d1_sub = d1[['date', 'macro_bullish', 'macro_bearish', 'macro_bull_alive', 'macro_bear_alive']]
    df = pd.merge(h1.reset_index(), d1_sub, on='date', how='left')
    df.ffill(inplace=True)
    df.set_index('datetime', inplace=True)
    
    # 4. Standardisasi format kolom kapital sesuai spesifikasi Backtesting.py
    col_rename = {'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close', 'volume': 'Volume'}
    df.rename(columns=col_rename, inplace=True)
    
    # 5. Feature Engineering Dimensi Mikro 1H
    df['EMA_micro'] = calc_ema_series(df['Close'], GLOBAL_PARAMS['ema_micro'])
    df['ATR'] = calc_atr_series(df['High'], df['Low'], df['Close'], GLOBAL_PARAMS['atr_period'])
    df.dropna(inplace=True)
    
    return df

print("Data Preprocessing Engine berhasil didefinisikan.")


Data Preprocessing Engine berhasil didefinisikan.


## Panduan Detail Alur Entry dan Exit Strategi (Cell 4: SymmetricMTFStrategy)
Seluruh aturan perdagangan (*rule-based quantitative rules*) yang dieksekusi oleh kelas strategi `SymmetricMTFStrategy` dirancang secara simetris dengan pembagian tahapan berikut:

---

### 1. Masuk Ketika Kondisinya Belum Apa (Prasyarat & Status Portofolio / Pre-Conditions)
Sebelum indikator teknikal dievaluasi untuk mengeksekusi order baru, algoritma memeriksa batasan operasional akun. Sistem **hanya boleh masuk jika seluruh kondisi berikut terpenuhi**:
1. **Portofolio Belum Memiliki Posisi Aktif (*Flat State*):**
   * Kondisi kode: `not self.position`
   * Akun wajib dalam keadaan tunai 100% (tidak sedang memegang posisi Long maupun Short). Algoritma tidak mengizinkan *averaging-down*, *pyramiding* (penambahan ukuran lot pada posisi yang berjalan), maupun *hedging* (membuka Long dan Short secara simultan).
2. **Masa Jeda Transaksi Belum Selesai Tidak Boleh Masuk (*Cooldown Enforcement*):**
   * Kondisi kode: `self.bars_since_exit >= self.cooldown_bars` (minimal 10 bar bursa = 10 jam perdagangan reguler atau ~1.5 hari bursa).
   * Jika posisi baru saja ditutup (baik terkena cut loss maupun profit lock), sistem dilarang membuka transaksi baru sebelum jeda 10 bar terlewati. Batasan ini menekan bias psikologis *revenge trading* dan mencegah kerugian berturut-turut pada fase konsolidasi/whipsaw.
3. **Periode Pemanasan Data Belum Mencukupi Tidak Boleh Masuk (*Warm-up Period*):**
   * Kondisi kode: `len(self.data) >= 16`
   * Algoritma menolak eksekusi jika bar per jam yang diproses masih di bawah periode EMA Mikro (16 bar) untuk menghindari sinyal palsu akibat kalkulasi rata-rata yang belum konvergen.
4. **Mode Perdagangan Belum Mengizinkan Tidak Boleh Masuk (*Execution Mode Filter*):**
   * Transaksi Beli (Long) hanya dievaluasi jika parameter mode diset ke `'long_only'` atau `'combined'`.
   * Transaksi Jual (Short) hanya dievaluasi jika parameter mode diset ke `'short_only'` atau `'combined'`.

---

### 2. Syarat Indikatornya Apa (Technical Indicator Entry Rules)

#### A. Sinyal Masuk Posisi Beli (LONG ENTRY)
Posisi Long dieksekusi (`self.buy(size=0.9999)`) pada harga penutupan bar saat ini (`Close[-1]`) jika dan hanya jika **seluruh 4 syarat teknikal berikut terpenuhi secara simultan**:
1. **Filter Tren Makro Harian (1D Bullish Regime):**
   * `self.macro_bullish[-1] == True`
   * Rumus: $	ext{EMA}_{21}(1	ext{D}) > 	ext{EMA}_{63}(1	ext{D}) 	imes (1 + 0.002)$
   * Garis tren cepat harian berada minimal 0.2% di atas garis tren lambat harian pada data kemarin (*yesterday's close*). Hal ini mengonfirmasi bahwa pasar berada dalam rezim *bullish* yang kuat secara struktural.
2. **Koreksi / Pullback Retest pada Timeframe 1H (Bar-2):**
   * `self.data.Low[-2] <= self.ema_micro[-2]`
   * Dua jam yang lalu, harga terendah (*Low*) sempat menyentuh atau menembus ke bawah garis rata-rata harga wajar mikro (EMA 16 jam). Ini membuktikan bahwa fase diskon/koreksi sehat sedang terjadi di dalam tren naik.
3. **Konfirmasi Pemantulan / Rebound pada Timeframe 1H (Bar-1):**
   * `self.data.Close[-1] > self.ema_micro[-1]`
   * Satu jam yang lalu, harga penutupan (*Close*) berhasil kembali ditutup di atas garis EMA Mikro 16 jam. Ini menandakan kembalinya minat beli dan penolakan terhadap harga murah.
4. **Konfirmasi Kemiringan Garis Mikro (*Slope Confirmation*):**
   * `self.ema_micro[-1] >= self.ema_micro[-2]`
   * Nilai EMA Mikro 16 jam saat ini tidak lebih rendah dari 1 jam sebelumnya. Memastikan bahwa momentum pelemahan mikro telah mereda dan garis moving average mulai mendatar atau melengkung ke atas.
5. **Penetapan Garis Risiko Awal (*Initial Stop Loss*):**
   * Seketika saat posisi Long dibuka, algoritma memasang garis pengaman awal sebesar 9.6 kali nilai ATR 12 jam di bawah harga beli:
     $$	ext{Trailing\_SL} = 	ext{EntryPrice} - (9.6 	imes 	ext{ATR}_{12})$$

#### B. Sinyal Masuk Posisi Jual (SHORT ENTRY)
Posisi Short dieksekusi (`self.sell(size=0.9999)`) pada harga penutupan bar saat ini (`Close[-1]`) jika dan hanya jika **seluruh 4 syarat teknikal berikut terpenuhi secara simultan**:
1. **Filter Tren Makro Harian (1D Bearish Regime):**
   * `self.macro_bearish[-1] == True`
   * Rumus: $	ext{EMA}_{21}(1	ext{D}) < 	ext{EMA}_{63}(1	ext{D}) 	imes (1 - 0.002)$
   * Garis tren cepat harian berada minimal 0.2% di bawah garis tren lambat harian pada data kemarin. Mengonfirmasi instrumen berada dalam rezim *bearish* yang dominan.
2. **Koreksi Naik / Pullback Retest Atas pada Timeframe 1H (Bar-2):**
   * `self.data.High[-2] >= self.ema_micro[-2]`
   * Dua jam yang lalu, harga tertinggi (*High*) sempat menguji atau menembus ke atas garis EMA Mikro 16 jam.
3. **Konfirmasi Penolakan Turun / Rejection pada Timeframe 1H (Bar-1):**
   * `self.data.Close[-1] < self.ema_micro[-1]`
   * Satu jam yang lalu, harga penutupan (*Close*) tertolak dan ditutup kembali di bawah garis EMA Mikro 16 jam. Menandakan dominasi tekanan jual jangka pendek.
4. **Konfirmasi Kemiringan Garis Mikro (*Slope Confirmation*):**
   * `self.ema_micro[-1] <= self.ema_micro[-2]`
   * Nilai EMA Mikro 16 jam saat ini lebih rendah atau sama dengan 1 jam sebelumnya, mengonfirmasi arah turunan momentum masih mengarah ke bawah.
5. **Penetapan Garis Risiko Awal (*Initial Stop Loss*):**
   * Seketika saat posisi Short dibuka, algoritma memasang garis pengaman awal sebesar 9.6 kali nilai ATR 12 jam di atas harga jual:
     $$	ext{Trailing\_SL} = 	ext{EntryPrice} + (9.6 	imes 	ext{ATR}_{12})$$

---

### 3. Mekanisme Pengunci Laba Berjalan (Trailing Stop / Ratchet Mechanism)
Ketika posisi sedang aktif berjalan, sistem mengevaluasi harga setiap penutupan candle 1 jam untuk memperketat proteksi keuntungan:
* **Ambang Batas Keaktifan (+2.3% Floating Profit):**
  * Selama posisi belum mencatat floating profit minimal **2.3%** (`Close[-1] < EntryPrice * 1.023` untuk Long, atau `Close[-1] > EntryPrice * 0.977` untuk Short), garis `Trailing_SL` **tetap diam di tempat** sebagai *Initial Cut Loss* guna memberi ruang napas dari noise intraday.
* **Pergeseran Ratchet untuk Posisi LONG:**
  * Begitu keuntungan belum terealisasi mencapai $\ge 2.3\%$, sistem menghitung kandidat batas baru: $	ext{new\_sl} = 	ext{Close}[-1] - (9.6 	imes 	ext{ATR})$.
  * Garis stop loss dinaikkan mengikuti titik tertinggi baru:
    $$	ext{Trailing\_SL} = \max(	ext{Trailing\_SL}, 	ext{new\_sl})$$
  * Garis stop loss Long tidak pernah diperbolehkan turun.
* **Pergeseran Ratchet untuk Posisi SHORT:**
  * Begitu keuntungan belum terealisasi mencapai $\ge 2.3\%$, sistem menghitung kandidat batas baru: $	ext{new\_sl} = 	ext{Close}[-1] + (9.6 	imes 	ext{ATR})$.
  * Garis stop loss diturunkan mengikuti titik terendah baru:
    $$	ext{Trailing\_SL} = \min(	ext{Trailing\_SL}, 	ext{new\_sl})$$
  * Garis stop loss Short tidak pernah diperbolehkan naik.

---

### 4. Keluar Ketika Apa (Kriteria Penutupan Posisi / Exit Rules)
Posisi yang sedang aktif **akan ditutup seketika (`self.position.close()`)** apabila memenuhi **salah satu dari 2 kondisi keluar mutlak berikut**:
1. **Kondisi Keluar 1: Penetrasi Trailing Stop / Initial Stop Loss (`Trailing Stop Hit`):**
   * **Posisi Long:** Harga penutupan per jam menembus ke bawah atau sama dengan garis stop loss (`Close[-1] <= self.trailing_sl`).
   * **Posisi Short:** Harga penutupan per jam menembus ke atas atau sama dengan garis stop loss (`Close[-1] >= self.trailing_sl`).
   * *Perilaku:* Jika posisi belum pernah mencapai keuntungan +2.3%, eksekusi ini berfungsi sebagai pemotong kerugian (*Cut Loss*). Jika posisi sudah berada dalam keuntungan besar, eksekusi ini berfungsi mengunci laba yang telah diraih (*Profit Lock*).
2. **Kondisi Keluar 2: Pembalikan Rezim Tren Makro Harian (`Macro Trend Reversal`):**
   * **Posisi Long:** Garis EMA 21 harian resmi berbalik memotong ke bawah garis EMA 63 harian (`not self.macro_bull_alive[-1]`).
   * **Posisi Short:** Garis EMA 21 harian resmi berbalik memotong ke atas garis EMA 63 harian (`not self.macro_bear_alive[-1]`).
   * *Perilaku:* Menutup posisi secara diskresioner karena landasan struktural tren besar (1D) yang menjadi prasyarat masuk telah gugur.

**Tindakan Pasca-Keluar (*Post-Exit Reset*):**
Seketika setelah transaksi ditutup oleh Kondisi Keluar 1 maupun 2, variabel `self.bars_since_exit` di-reset menjadi `0`. Ini mengunci sistem agar tidak dapat melakukan entry baru selama **10 bar perdagangan berikutnya** (*cooldown period*).

In [24]:
# =============================================================================
# CELL 4: UNIFIED STRATEGY CLASS (SymmetricMTFStrategy)
# =============================================================================
class SymmetricMTFStrategy(Strategy):
    """
    Strategi Kuantitatif Multi-Timeframe Simetris Penuh.
    Mendukung mode 'long_only', 'short_only', dan 'combined'.
    """
    mode = 'combined' # 'long_only', 'short_only', atau 'combined'
    atr_mult = GLOBAL_PARAMS['atr_mult']
    cooldown_bars = GLOBAL_PARAMS['cooldown_bars']
    min_profit_pct = GLOBAL_PARAMS['min_profit_pct']

    def init(self):
        # Indikator teknikal
        self.ema_micro = self.I(calc_ema_series, self.data.Close, GLOBAL_PARAMS['ema_micro'], overlay=True)
        self.atr = self.I(calc_atr_series, self.data.High, self.data.Low, self.data.Close, GLOBAL_PARAMS['atr_period'], overlay=False)
        self.macro_bullish = self.I(lambda: self.data.df['macro_bullish'].values, overlay=False)
        self.macro_bearish = self.I(lambda: self.data.df['macro_bearish'].values, overlay=False)
        self.macro_bull_alive = self.I(lambda: self.data.df['macro_bull_alive'].values, overlay=False)
        self.macro_bear_alive = self.I(lambda: self.data.df['macro_bear_alive'].values, overlay=False)

        # Variabel pelacak internal
        self.bars_since_exit = 999
        self.trailing_sl = 0.0
        self.entry_price = 0.0
        self.trade_records = []
        self.active_record = None

    def next(self):
        # Tahap 0: Validasi Warm-up Period
        if len(self.data) < GLOBAL_PARAMS['ema_micro']:
            return

        # Tahap 1: Evaluasi Entry jika Portofolio Kosong
        if not self.position:
            self.bars_since_exit += 1
            if self.bars_since_exit >= self.cooldown_bars:
                long_allowed = (self.mode in ['long_only', 'combined'])
                short_allowed = (self.mode in ['short_only', 'combined'])

                long_trigger = (
                    long_allowed and
                    self.macro_bullish[-1] and
                    (self.data.Low[-2] <= self.ema_micro[-2]) and
                    (self.data.Close[-1] > self.ema_micro[-1]) and
                    (self.ema_micro[-1] >= self.ema_micro[-2])
                )

                short_trigger = (
                    short_allowed and
                    self.macro_bearish[-1] and
                    (self.data.High[-2] >= self.ema_micro[-2]) and
                    (self.data.Close[-1] < self.ema_micro[-1]) and
                    (self.ema_micro[-1] <= self.ema_micro[-2])
                )

                if long_trigger:
                    self.buy(size=0.9999)
                    self.entry_price = self.data.Close[-1]
                    self.trailing_sl = self.entry_price - (self.atr_mult * self.atr[-1])
                    self.active_record = {
                        'EntryBar': len(self.data) - 1,
                        'EntryTime': self.data.index[-1],
                        'Mode': self.mode,
                        'Position_Type': 'LONG',
                        'ATR_At_Entry': self.atr[-1],
                        'Trailing_SL_Final': self.trailing_sl,
                        'Exit_Reason': None
                    }
                    return

                elif short_trigger:
                    self.sell(size=0.9999)
                    self.entry_price = self.data.Close[-1]
                    self.trailing_sl = self.entry_price + (self.atr_mult * self.atr[-1])
                    self.active_record = {
                        'EntryBar': len(self.data) - 1,
                        'EntryTime': self.data.index[-1],
                        'Mode': self.mode,
                        'Position_Type': 'SHORT',
                        'ATR_At_Entry': self.atr[-1],
                        'Trailing_SL_Final': self.trailing_sl,
                        'Exit_Reason': None
                    }
                    return

        # Tahap 2 & 3: Manajemen Posisi Berjalan (Ratchet SL & Exit Rules)
        else:
            if self.position.is_long:
                # Ratchet Trailing Stop
                is_in_profit = self.data.Close[-1] >= (self.entry_price * (1.0 + self.min_profit_pct))
                if is_in_profit:
                    new_sl = self.data.Close[-1] - (self.atr_mult * self.atr[-1])
                    self.trailing_sl = max(self.trailing_sl, new_sl)
                if self.active_record:
                    self.active_record['Trailing_SL_Final'] = self.trailing_sl

                # Exit 1: Trailing Stop Hit
                if self.data.Close[-1] <= self.trailing_sl:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Trailing Stop Hit'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

                # Exit 2: Macro Trend Reversal
                if not self.macro_bull_alive[-1]:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Macro Trend Reversal'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

            elif self.position.is_short:
                # Ratchet Trailing Stop
                is_in_profit = self.data.Close[-1] <= (self.entry_price * (1.0 - self.min_profit_pct))
                if is_in_profit:
                    new_sl = self.data.Close[-1] + (self.atr_mult * self.atr[-1])
                    self.trailing_sl = min(self.trailing_sl, new_sl)
                if self.active_record:
                    self.active_record['Trailing_SL_Final'] = self.trailing_sl

                # Exit 1: Trailing Stop Hit
                if self.data.Close[-1] >= self.trailing_sl:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Trailing Stop Hit'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

                # Exit 2: Macro Trend Reversal
                if not self.macro_bear_alive[-1]:
                    if self.active_record:
                        self.active_record['Exit_Reason'] = 'Macro Trend Reversal'
                        self.trade_records.append(self.active_record.copy())
                        self.active_record = None
                    self.position.close()
                    self.bars_since_exit = 0
                    return

print("Unified Strategy Class (SymmetricMTFStrategy) berhasil didefinisikan.")

Unified Strategy Class (SymmetricMTFStrategy) berhasil didefinisikan.


## Cell 5: Rekonsiliasi Transaksi Bebas Look-Behind Bias (reconcile_trades)
Menggabungkan log transaksi internal `Backtesting.py` dengan pencatat status strategi. Menghitung *Maximum Adverse Excursion* (MAE), *Maximum Favorable Excursion* (MFE), dan *Profit Giveback* murni dari bar setelah eksekusi (`entry_bar + 1` hingga `exit_bar`) guna mengeliminasi bias harga yang terjadi sebelum eksekusi pada harga *Close*.

In [25]:
# =============================================================================
# CELL 5: TRADE RECONCILIATION & CLEANING FUNCTION
# =============================================================================
def reconcile_trades(stats, ohlc_df, mode_label, fractional_unit=1e-4):

    trades = stats['_trades'].copy()
    if len(trades) == 0:
        return pd.DataFrame(columns=[
            'Trade_ID', 'Mode', 'Position_Type', 'Size', 'EntryTime', 'ExitTime',
            'Duration', 'Bars_Held', 'EntryPrice', 'ExitPrice', 'Trailing_SL_Final',
            'ATR_At_Entry', 'PnL_USD', 'Return_Pct', 'MAE_Pct', 'MFE_Pct',
            'Profit_Giveback_Pct', 'Exit_Reason'
        ])
    
    strat = stats['_strategy']
    records = list(strat.trade_records)
    
    # Tangani transaksi yang ditutup otomatis pada akhir rentang data oleh engine
    if strat.active_record is not None and len(records) < len(trades):
        rec_copy = strat.active_record.copy()
        rec_copy['Exit_Reason'] = 'End of Data'
        records.append(rec_copy)
        
    trade_rows = []
    
    for i in range(len(trades)):
        tr = trades.iloc[i]
        entry_bar = int(tr['EntryBar'])
        exit_bar = int(tr['ExitBar'])
        pos_size = float(tr['Size'])
        is_long = pos_size > 0
        pos_type = 'LONG' if is_long else 'SHORT'
        
        entry_px = float(tr['EntryPrice'])
        exit_px = float(tr['ExitPrice'])
        pnl = float(tr['PnL'])
        ret_pct = float(tr['ReturnPct']) * 100.0
        
        bars_held = exit_bar - entry_bar
        
        # Eliminasi Look-Behind Bias: Evaluasi MAE/MFE murni pada bar setelah entri Close
        if exit_bar > entry_bar:
            trade_slice = ohlc_df.iloc[entry_bar + 1 : exit_bar + 1]
        else:
            trade_slice = ohlc_df.iloc[entry_bar : entry_bar + 1]
        
        if is_long:
            lowest_price = float(trade_slice['Low'].min())
            mae_usd = min(0.0, lowest_price - entry_px)
            mae_pct = (mae_usd / entry_px) * 100.0
            
            highest_price = float(trade_slice['High'].max())
            mfe_usd = max(0.0, highest_price - entry_px)
            mfe_pct = (mfe_usd / entry_px) * 100.0
            
            if highest_price > entry_px:
                profit_giveback_pct = ((exit_px - highest_price) / highest_price) * 100.0
            else:
                profit_giveback_pct = 0.0
        else:
            highest_price = float(trade_slice['High'].max())
            mae_usd = min(0.0, entry_px - highest_price)
            mae_pct = (mae_usd / entry_px) * 100.0
            
            lowest_price = float(trade_slice['Low'].min())
            mfe_usd = max(0.0, entry_px - lowest_price)
            mfe_pct = (mfe_usd / entry_px) * 100.0
            
            if lowest_price < entry_px:
                profit_giveback_pct = ((lowest_price - exit_px) / lowest_price) * 100.0
            else:
                profit_giveback_pct = 0.0
                
        # Rekonsiliasi variabel internal pelacak
        if i < len(records):
            r = records[i]
            sl_final = r['Trailing_SL_Final'] / fractional_unit if fractional_unit > 0 else r['Trailing_SL_Final']
            atr_entry = r['ATR_At_Entry'] / fractional_unit if fractional_unit > 0 else r['ATR_At_Entry']
            exit_reason = r['Exit_Reason'] or ('End of Data' if exit_bar >= len(ohlc_df) - 1 else 'Trailing Stop Hit')
        else:
            sl_final = entry_px
            atr_entry = float(ohlc_df['ATR'].iloc[entry_bar])
            exit_reason = 'End of Data' if exit_bar >= len(ohlc_df) - 1 else 'Trailing Stop Hit'
            
        trade_rows.append({
            'Trade_ID': i + 1,
            'Mode': mode_label,
            'Position_Type': pos_type,
            'Size': round(pos_size, 4),
            'EntryTime': pd.to_datetime(tr['EntryTime']).strftime('%Y-%m-%d %H:%M'),
            'ExitTime': pd.to_datetime(tr['ExitTime']).strftime('%Y-%m-%d %H:%M'),
            'Duration': str(tr['Duration']),
            'Bars_Held': bars_held,
            'EntryPrice': round(entry_px, 2),
            'ExitPrice': round(exit_px, 2),
            'Trailing_SL_Final': round(sl_final, 2),
            'ATR_At_Entry': round(atr_entry, 4),
            'PnL_USD': round(pnl, 2),
            'Return_Pct': round(ret_pct, 2),
            'MAE_Pct': round(mae_pct, 2),
            'MFE_Pct': round(mfe_pct, 2),
            'Profit_Giveback_Pct': round(profit_giveback_pct, 2),
            'Exit_Reason': exit_reason
        })
        
    return pd.DataFrame(trade_rows)

print("Fungsi Rekonsiliasi Transaksi (reconcile_trades) berhasil didefinisikan.")


Fungsi Rekonsiliasi Transaksi (reconcile_trades) berhasil didefinisikan.


## Cell 6: Ekstraksi Metrik Standar Industri & Komparasi Apple-to-Apple
Mengekstrak 10 metrik performa kuantitatif: Total Return (%), CAGR (%), Max Drawdown (%), Calmar Ratio, Sharpe Ratio (faktor tahunan 252 hari), Sortino Ratio (*Target Downside Deviation* = 0), Win Rate (%), Profit Factor, Total Trades, dan Exposure Time (%). Menjamin formula matematis yang identik antara strategi aktif dan *benchmark* pasif Buy & Hold.

In [26]:
# =============================================================================
# CELL 6: METRIC EXTRACTION & BUY & HOLD BENCHMARK
# =============================================================================
def extract_metrics(stats, df, is_bnh=False):
    """
    Ekstraksi metrik kuantitatif terstandarisasi dan apple-to-apple:
      - Total Return (%)
      - CAGR (%)
      - Max Drawdown (%)
      - Calmar Ratio
      - Sharpe Ratio
      - Sortino Ratio
      - Win Rate (%)
      - Profit Factor
      - Total Trades
      - Exposure Time (%)
    """
    dur_years = (df.index[-1] - df.index[0]).total_seconds() / (365.25 * 86400)

    if is_bnh:
        tot_ret = float(stats['Buy & Hold Return [%]'])
        cagr = ((1.0 + tot_ret / 100.0) ** (1.0 / dur_years) - 1.0) * 100.0 if tot_ret > -100.0 else -100.0

        equity_series = df['Close'] / df['Close'].iloc[0]
        mdd = float(((equity_series - equity_series.cummax()) / equity_series.cummax()).min() * 100.0)

        daily_ret = df['Close'].resample('D').last().dropna().pct_change().dropna()
        wr = np.nan
        pf = np.nan
        n_trades = 1
        exposure = 100.0
    else:
        tot_ret = float(stats['Return [%]'])
        cagr = float(stats['Return (Ann.) [%]'])
        mdd = float(stats['Max. Drawdown [%]'])

        # Gunakan kurva ekuitas riil strategi untuk resampling harian yang identik dengan BnH
        eq_curve = stats['_equity_curve']['Equity']
        daily_ret = eq_curve.resample('D').last().dropna().pct_change().dropna()

        wr = float(stats['Win Rate [%]']) if not pd.isna(stats['Win Rate [%]']) else 0.0
        pf = float(stats['Profit Factor']) if not pd.isna(stats['Profit Factor']) else 0.0
        n_trades = int(stats['# Trades'])
        exposure = float(stats['Exposure Time [%]']) if not pd.isna(stats['Exposure Time [%]']) else 0.0

    # Perhitungan Sharpe, Sortino (Target Downside Deviation = 0), dan Calmar yang seragam
    if len(daily_ret) > 1 and daily_ret.std() > 0:
        sharpe = float((daily_ret.mean() / daily_ret.std()) * np.sqrt(252))
    else:
        sharpe = 0.0

    downside_diff = np.minimum(0.0, daily_ret)
    downside_std = np.sqrt(np.mean(downside_diff ** 2))
    if len(daily_ret) > 1 and downside_std > 0:
        sortino = float((daily_ret.mean() / downside_std) * np.sqrt(252))
    else:
        sortino = 0.0

    calmar = float(cagr / abs(mdd)) if abs(mdd) > 0 else 0.0

    return pd.Series({
        'Total Return (%)': tot_ret,
        'CAGR (%)': cagr,
        'Max Drawdown (%)': mdd,
        'Calmar Ratio': calmar,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Win Rate (%)': wr,
        'Profit Factor': pf,
        'Total Trades': n_trades,
        'Exposure Time (%)': exposure
    })

print("Fungsi Ekstraksi Metrik Kuantitatif (extract_metrics) berhasil didefinisikan.")

Fungsi Ekstraksi Metrik Kuantitatif (extract_metrics) berhasil didefinisikan.


## Cell 7: Loop Eksekusi Batch Lintas Instrumen
Mengeksekusi simulasi secara berurutan untuk keempat instrumen acuan (`SPY`, `QQQ`, `DJI`, `AAPL`). Setiap instrumen diuji pada 4 skenario terpisah:
1. Benchmark Pasif: **Buy & Hold**
2. Strategi Aktif Searah: **Long-Only**
3. Strategi Aktif Searah: **Short-Only**
4. Strategi Aktif Dua Arah Riil: **Combined**

Memvalidasi bahwa data transaksi bebas dari nilai `NaN` (*Zero-NaN Guarantee*) serta memenuhi spesifikasi format 18 kolom.

In [27]:
# =============================================================================
# CELL 7: BATCH EXECUTION LOOP
# =============================================================================
ALL_PERFORMANCE_TABLES = {}
ALL_TRADES_TABLES = {}

print("=" * 80)
print(f"MEMULAI SIMULASI BACKTEST BATCH PADA INSTRUMEN: {ASSETS}")
print("=" * 80)

for asset in ASSETS:
    print(f"\n>>> Memproses Instrumen: [{asset}]")
    df = preprocess_asset_data(asset)
    print(f"    Data siap diuji: {len(df):,} bar 1H ({df.index[0]} s.d. {df.index[-1]})")
    
    # 1. Eksekusi Backtest Mode Long-Only
    bt_long = FractionalBacktest(
        df,
        SymmetricMTFStrategy,
        cash=GLOBAL_PARAMS['initial_capital'],
        commission=GLOBAL_PARAMS['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )
    stats_long = bt_long.run(mode='long_only')
    trades_long = reconcile_trades(stats_long, df, 'Long-Only')
    s_long = extract_metrics(stats_long, df, is_bnh=False)
    
    # 2. Eksekusi Evaluasi Buy & Hold Benchmark
    s_bnh = extract_metrics(stats_long, df, is_bnh=True)
    
    # 3. Eksekusi Backtest Mode Short-Only
    bt_short = FractionalBacktest(
        df,
        SymmetricMTFStrategy,
        cash=GLOBAL_PARAMS['initial_capital'],
        commission=GLOBAL_PARAMS['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )
    stats_short = bt_short.run(mode='short_only')
    trades_short = reconcile_trades(stats_short, df, 'Short-Only')
    s_short = extract_metrics(stats_short, df, is_bnh=False)
    
    # 4. Eksekusi Backtest Mode Combined (Dua Arah Riil)
    bt_comb = FractionalBacktest(
        df,
        SymmetricMTFStrategy,
        cash=GLOBAL_PARAMS['initial_capital'],
        commission=GLOBAL_PARAMS['fee'],
        trade_on_close=True,
        finalize_trades=True,
        fractional_unit=1e-4
    )
    stats_comb = bt_comb.run(mode='combined')
    trades_comb = reconcile_trades(stats_comb, df, 'Combined')
    s_comb = extract_metrics(stats_comb, df, is_bnh=False)
    
    # 5. Penyusunan Tabel Ringkasan Kinerja (Baris = Metrik, Kolom = Mode Pengujian)
    df_perf = pd.DataFrame({
        'Buy & Hold': s_bnh,
        'Long-Only': s_long,
        'Short-Only': s_short,
        'Combined': s_comb
    })
    ALL_PERFORMANCE_TABLES[asset] = df_perf
    
    # 6. Penggabungan Seluruh Log Transaksi Bersih untuk Instrumen Ini
    df_trades_all = pd.concat([trades_long, trades_short, trades_comb], ignore_index=True)
    df_trades_all['Trade_ID'] = range(1, len(df_trades_all) + 1)
    
    # Validasi Integritas Data (Zero NaN Guarantee)
    assert df_trades_all.isna().sum().sum() == 0, f"Ditemukan nilai NaN pada trades {asset}"
    assert len(df_trades_all.columns) == 18, f"Jumlah kolom tidak sesuai standar 18 kolom pada {asset}"
    
    ALL_TRADES_TABLES[asset] = df_trades_all
    
    print(f"    Simulasi [{asset}] selesai | Total Trades: Long={len(trades_long)}, Short={len(trades_short)}, Combined={len(trades_comb)} (Total: {len(df_trades_all)}, NaNs: 0)")

print("\n" + "=" * 80)
print("SELURUH SIMULASI BATCH INSTRUMEN BERHASIL DISELESAIKAN!")
print("=" * 80)

MEMULAI SIMULASI BACKTEST BATCH PADA INSTRUMEN: ['SPY', 'QQQ', 'DJI', 'AAPL']

>>> Memproses Instrumen: [SPY]
    Data siap diuji: 12,829 bar 1H (2018-02-19 16:00:00 s.d. 2024-05-10 16:00:00)


FractionalBacktest.run:   0%|          | 0/12828 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/12828 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/12828 [00:00<?, ?bar/s]

    Simulasi [SPY] selesai | Total Trades: Long=57, Short=20, Combined=77 (Total: 154, NaNs: 0)

>>> Memproses Instrumen: [QQQ]
    Data siap diuji: 12,769 bar 1H (2018-02-19 09:00:00 s.d. 2024-05-10 16:00:00)


FractionalBacktest.run:   0%|          | 0/12768 [00:00<?, ?bar/s]

... (7 outputs left out)

FractionalBacktest.run:   0%|          | 0/11879 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/11879 [00:00<?, ?bar/s]

FractionalBacktest.run:   0%|          | 0/11879 [00:00<?, ?bar/s]

    Simulasi [AAPL] selesai | Total Trades: Long=19, Short=14, Combined=33 (Total: 66, NaNs: 0)

SELURUH SIMULASI BATCH INSTRUMEN BERHASIL DISELESAIKAN!


## Cell 8: Tampilan Tabel Kinerja Komparatif Notebook
Merender tabel ringkasan kinerja per instrumen secara visual di dalam notebook dengan format bilangan desimal dua angka di belakang koma untuk kemudahan presentasi saat sidang skripsi.

In [28]:
# =============================================================================
# CELL 8: NOTEBOOK DISPLAY
# =============================================================================
from IPython.display import display

for asset in ASSETS:
    print("\n" + "=" * 85)
    print(f"TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - {asset}")
    print("=" * 85)
    
    df_display = ALL_PERFORMANCE_TABLES[asset].copy()
    
    # Format representasi angka rapi
    formatted_df = df_display.copy()
    for col in formatted_df.columns:
        formatted_df[col] = formatted_df[col].apply(
            lambda v: "-" if pd.isna(v) else (f"{int(v)}" if abs(v - round(v)) < 1e-6 and v >= 1 else f"{v:.2f}")
        )
        
    display(formatted_df)



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - SPY


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),91.68,43.00,-19.22,15.51
CAGR (%),11.03,5.77,-3.29,2.29
Max Drawdown (%),-34.67,-26.32,-34.12,-35.55
Calmar Ratio,0.32,0.22,-0.10,0.06
Sharpe Ratio,0.66,0.57,-0.23,0.22
Sortino Ratio,0.91,0.79,-0.34,0.32
Win Rate (%),-,61.40,40,55.84
Profit Factor,-,1.65,0.65,1.18
Total Trades,1,57,20,77
Exposure Time (%),100,63.86,21.62,85.48



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - QQQ


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),167.51,85.33,-29.52,51.75
CAGR (%),17.14,10.22,-5.37,6.80
Max Drawdown (%),-37.25,-23.69,-36.54,-28.93
Calmar Ratio,0.46,0.43,-0.15,0.23
Sharpe Ratio,0.80,0.75,-0.33,0.43
Sortino Ratio,1.14,1.04,-0.44,0.59
Win Rate (%),-,63.16,25,54.43
Profit Factor,-,2.06,0.56,1.42
Total Trades,1,57,24,79
Exposure Time (%),100,64.33,20.50,84.40



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - DJI


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),56.42,16.36,-35.03,-16.74
CAGR (%),7.46,2.40,-6.53,-2.83
Max Drawdown (%),-37.63,-17.51,-43.06,-29.83
Calmar Ratio,0.20,0.14,-0.15,-0.09
Sharpe Ratio,0.48,0.28,-0.50,-0.11
Sortino Ratio,0.67,0.39,-0.72,-0.15
Win Rate (%),-,60.78,28.57,52.05
Profit Factor,-,1.32,0.37,0.89
Total Trades,1,51,21,73
Exposure Time (%),100,61.31,25.04,85.91



TABEL RINGKASAN KINERJA SIMETRIS MULTI-TIMEFRAME - AAPL


,Buy & Hold,Long-Only,Short-Only,Combined
Total Return (%),327.63,101.01,-57.29,-24.97
CAGR (%),26.31,12.59,-13.45,-4.76
Max Drawdown (%),-37.42,-32.21,-59.73,-48.60
Calmar Ratio,0.70,0.39,-0.23,-0.10
Sharpe Ratio,1.15,0.75,-0.98,-0.10
Sortino Ratio,1.67,1.11,-1.33,-0.14
Win Rate (%),-,36.84,14.29,27.27
Profit Factor,-,2.71,0.11,1.02
Total Trades,1,19,14,33
Exposure Time (%),100,68.64,27.68,95.22


## Cell 9: Ekspor Berkas Excel Terpisah (Dual Excel Exporter)
Menyimpan seluruh luaran simulasi ke dalam 2 berkas Excel terpisah menggunakan pustaka `openpyxl`:
1. `Ringkasan_Kinerja_Semua_Instrumen.xlsx`: 4 lembar kerja (satu sheet per instrumen).
2. `Detail_Trades_Semua_Instrumen.xlsx`: 4 lembar kerja berisi seluruh log transaksi 18 kolom institusional.

In [29]:
# =============================================================================
# CELL 9: DUAL EXCEL EXPORTER (openpyxl)
# =============================================================================
# Inisialisasi Workbook Excel Terpisah
wb_ringkasan = openpyxl.Workbook()
wb_ringkasan.remove(wb_ringkasan.active) # Hapus sheet default

wb_trades = openpyxl.Workbook()
wb_trades.remove(wb_trades.active) # Hapus sheet default

# Gaya Visual Terstandarisasi (Institutional Clean Theme)
header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
cell_font = Font(name="Calibri", size=10)
bold_font = Font(name="Calibri", size=10, bold=True)
thin_border = Border(
    left=Side(style='thin', color='D9D9D9'),
    right=Side(style='thin', color='D9D9D9'),
    top=Side(style='thin', color='D9D9D9'),
    bottom=Side(style='thin', color='D9D9D9')
)

# -----------------------------------------------------------------------------
# 1. Ekspor Berkas Ringkasan Kinerja (Ringkasan_Kinerja_Semua_Instrumen.xlsx)
# -----------------------------------------------------------------------------
for asset in ASSETS:
    ws = wb_ringkasan.create_sheet(title=asset)
    df_perf = ALL_PERFORMANCE_TABLES[asset]
    
    # Baris Judul Kolom
    headers = ['Metrik'] + df_perf.columns.tolist()
    ws.append(headers)
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border
        
    # Baris Data Metrik
    for row_idx, (metric_name, row_vals) in enumerate(df_perf.iterrows(), start=2):
        row_data = [metric_name]
        for val in row_vals:
            if pd.isna(val):
                row_data.append("-")
            elif metric_name == 'Total Trades':
                row_data.append(int(val))
            else:
                row_data.append(round(float(val), 2))
        ws.append(row_data)
        
        # Penataan Gaya Sel Data
        metric_cell = ws.cell(row=row_idx, column=1)
        metric_cell.font = bold_font
        metric_cell.border = thin_border
        
        for c_idx in range(2, len(row_data) + 1):
            c = ws.cell(row=row_idx, column=c_idx)
            c.font = cell_font
            c.alignment = Alignment(horizontal="right", vertical="center")
            c.border = thin_border
            if isinstance(c.value, (int, float)):
                c.number_format = '#,##0.00' if isinstance(c.value, float) else '#,##0'
                
    # Otomatisasi Penyesuaian Lebar Kolom
    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 4, 15)

wb_ringkasan.save(FILE_RINGKASAN)
print(f"Laporan 1 Berhasil Disimpan: {FILE_RINGKASAN}")

# -----------------------------------------------------------------------------
# 2. Ekspor Berkas Detail Transaksi (Detail_Trades_Semua_Instrumen.xlsx)
# -----------------------------------------------------------------------------
for asset in ASSETS:
    ws = wb_trades.create_sheet(title=f"Trades_{asset}")
    df_tr = ALL_TRADES_TABLES[asset]
    
    # Validasi Integritas Data Sebelum Ekspor
    assert df_tr.isna().sum().sum() == 0, f"Ditemukan nilai NaN pada trades {asset} sebelum ekspor"
    
    headers = df_tr.columns.tolist()
    ws.append(headers)
    for col_idx in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=col_idx)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border
        
    for row_idx, (_, tr_row) in enumerate(df_tr.iterrows(), start=2):
        ws.append(tr_row.tolist())
        for c_idx in range(1, len(headers) + 1):
            c = ws.cell(row=row_idx, column=c_idx)
            c.font = cell_font
            c.border = thin_border
            col_name = headers[c_idx - 1]
            
            if col_name in ['EntryPrice', 'ExitPrice', 'Trailing_SL_Final', 'PnL_USD', 'Return_Pct', 'MAE_Pct', 'MFE_Pct', 'Profit_Giveback_Pct']:
                c.alignment = Alignment(horizontal="right", vertical="center")
                if isinstance(c.value, (int, float)):
                    c.number_format = '#,##0.00'
            elif col_name in ['Size', 'ATR_At_Entry']:
                c.alignment = Alignment(horizontal="right", vertical="center")
                if isinstance(c.value, (int, float)):
                    c.number_format = '#,##0.0000'
            elif col_name in ['Trade_ID', 'Bars_Held']:
                c.alignment = Alignment(horizontal="center", vertical="center")
                if isinstance(c.value, (int, float)):
                    c.number_format = '#,##0'
            else:
                c.alignment = Alignment(horizontal="center", vertical="center")

    for col in ws.columns:
        max_len = max(len(str(cell.value or '')) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 14)

wb_trades.save(FILE_TRADES)
print(f"Laporan 2 Berhasil Disimpan: {FILE_TRADES}")
print("\nSeluruh proses ekspor Excel selesai.")

Laporan 1 Berhasil Disimpan: E:\kultivasi_4_tahun\ta\Ringkasan_Kinerja_Semua_Instrumen.xlsx
Laporan 2 Berhasil Disimpan: E:\kultivasi_4_tahun\ta\Detail_Trades_Semua_Instrumen.xlsx

Seluruh proses ekspor Excel selesai.


## Cell 10: Modul Optimasi Ruang Parameter Dinamis (Parallel Randomized Search)
*Catatan: Sel ini sengaja di-comment out secara default untuk keperluan eksekusi manual terpisah saat eksplorasi parameter Monte Carlo multi-core.*

In [30]:
# # =============================================================================
# # CELL 10: DYNAMIC RANGE & RANDOMIZED GRID SEARCH OPTIMIZATION (PARALLEL)
# # =============================================================================
# import os
# import sys
# import time
# import random
# import itertools
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from tqdm.auto import tqdm
# import concurrent.futures
# import openpyxl
# from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
# from openpyxl.utils import get_column_letter

# # Monkey-patch untuk kompatibilitas Backtesting.py di sub-proses
# pd.DataFrame.iteritems = pd.DataFrame.items

# # -----------------------------------------------------------------------------
# # 1. PEMBUATAN WORKER MODULE (UNTUK MULTIPROCESSING WINDOWS-SAFE)
# # -----------------------------------------------------------------------------
# worker_module_path = BASE_DIR / "optimization_worker.py"
# worker_script_content = """import os
# import sys
# import pandas as pd
# import numpy as np
# pd.DataFrame.iteritems = pd.DataFrame.items
# from backtesting import Strategy
# from backtesting.lib import FractionalBacktest

# def calc_ema_series(series, span):
#     return pd.Series(series).ewm(span=span, adjust=False).mean().values

# def calc_atr_series(high, low, close, period=14):
#     high_s = pd.Series(high)
#     low_s = pd.Series(low)
#     close_s = pd.Series(close)
#     close_prev = close_s.shift(1)
#     tr = pd.concat([
#         high_s - low_s,
#         (high_s - close_prev).abs(),
#         (low_s - close_prev).abs()
#     ], axis=1).max(axis=1)
#     return pd.Series(tr).ewm(span=period, adjust=False).mean().values

# class OptMTFStrategy(Strategy):
#     mode = 'long_only'
#     atr_mult = 6.0
#     cooldown_bars = 7
#     min_profit_pct = 0.010
#     ema_micro_span = 33
#     atr_period_span = 14

#     def init(self):
#         self.ema_micro = self.I(calc_ema_series, self.data.Close, self.ema_micro_span, overlay=True)
#         self.atr = self.I(calc_atr_series, self.data.High, self.data.Low, self.data.Close, self.atr_period_span, overlay=False)
#         self.macro_bullish = self.I(lambda: self.data.df['macro_bullish'].values, overlay=False)
#         self.macro_bull_alive = self.I(lambda: self.data.df['macro_bull_alive'].values, overlay=False)
#         self.bars_since_exit = 999
#         self.trailing_sl = 0.0
#         self.entry_price = 0.0

#     def next(self):
#         if len(self.data) < self.ema_micro_span:
#             return

#         if not self.position:
#             self.bars_since_exit += 1
#             if self.bars_since_exit >= self.cooldown_bars:
#                 long_trigger = (
#                     self.macro_bullish[-1] and
#                     (self.data.Low[-2] <= self.ema_micro[-2]) and
#                     (self.data.Close[-1] > self.ema_micro[-1]) and
#                     (self.ema_micro[-1] >= self.ema_micro[-2])
#                 )
#                 if long_trigger:
#                     self.buy(size=0.9999)
#                     self.entry_price = self.data.Close[-1]
#                     self.trailing_sl = self.entry_price - (self.atr_mult * self.atr[-1])
#         else:
#             is_in_profit = self.data.Close[-1] >= (self.entry_price * (1.0 + self.min_profit_pct))
#             if is_in_profit:
#                 new_sl = self.data.Close[-1] - (self.atr_mult * self.atr[-1])
#                 self.trailing_sl = max(self.trailing_sl, new_sl)

#             if self.data.Close[-1] <= self.trailing_sl:
#                 self.position.close()
#                 self.bars_since_exit = 0
#                 return

#             if not self.macro_bull_alive[-1]:
#                 self.position.close()
#                 self.bars_since_exit = 0
#                 return

# PRELOADED_DATA = {}

# def init_worker(preloaded_dict):
#     global PRELOADED_DATA
#     PRELOADED_DATA = preloaded_dict

# def evaluate_combination_task(combo_tuple):
#     p_spread, p_ema_micro, p_atr_period, p_atr_mult, p_cooldown, p_min_profit = combo_tuple
    
#     class TargetStrat(OptMTFStrategy):
#         atr_mult = p_atr_mult
#         cooldown_bars = p_cooldown
#         min_profit_pct = p_min_profit
#         ema_micro_span = p_ema_micro
#         atr_period_span = p_atr_period

#     asset_returns = {}
#     asset_mdds = {}
#     asset_trades = {}

#     for asset in ['SPY', 'QQQ', 'DJI']:
#         base_df = PRELOADED_DATA[asset].copy(deep=False)
#         base_df['macro_bullish'] = base_df['ema_fast'] > (base_df['ema_slow'] * (1.0 + p_spread))
        
#         bt = FractionalBacktest(
#             base_df,
#             TargetStrat,
#             cash=1000.0,
#             commission=0.0003,
#             trade_on_close=True,
#             finalize_trades=True,
#             fractional_unit=1e-4
#         )
#         stats = bt.run()
        
#         ret = float(stats['Return [%]'])
#         mdd = float(stats['Max. Drawdown [%]'])
#         tr = int(stats['# Trades'])
        
#         asset_returns[asset] = ret
#         asset_mdds[asset] = mdd
#         asset_trades[asset] = tr

#     avg_return = (asset_returns['SPY'] + asset_returns['QQQ'] + asset_returns['DJI']) / 3.0
#     avg_mdd = (asset_mdds['SPY'] + asset_mdds['QQQ'] + asset_mdds['DJI']) / 3.0

#     return {
#         'macro_spread_threshold': p_spread,
#         'ema_micro': p_ema_micro,
#         'atr_period': p_atr_period,
#         'atr_mult': p_atr_mult,
#         'cooldown_bars': p_cooldown,
#         'min_profit_pct': p_min_profit,
#         'Avg_Total_Return': round(avg_return, 2),
#         'Avg_Max_Drawdown': round(avg_mdd, 2),
#         'SPY_Return (%)': round(asset_returns['SPY'], 2),
#         'QQQ_Return (%)': round(asset_returns['QQQ'], 2),
#         'DJI_Return (%)': round(asset_returns['DJI'], 2),
#         'SPY_MaxDD (%)': round(asset_mdds['SPY'], 2),
#         'QQQ_MaxDD (%)': round(asset_mdds['QQQ'], 2),
#         'DJI_MaxDD (%)': round(asset_mdds['DJI'], 2),
#         'SPY_Trades': asset_trades['SPY'],
#         'QQQ_Trades': asset_trades['QQQ'],
#         'DJI_Trades': asset_trades['DJI'],
#     }
# """
# worker_module_path.write_text(worker_script_content, encoding='utf-8')
# sys.path.insert(0, str(BASE_DIR))
# import optimization_worker

# # -----------------------------------------------------------------------------
# # 2. PEMBUATAN DYNAMIC RANGE (NUMPY ARANGE & PYTHON RANGE)
# # -----------------------------------------------------------------------------
# macro_spread_threshold_range = np.round(np.arange(0.002, 0.025 + 0.0005, 0.001), 3).tolist()
# ema_micro_range              = list(range(14, 51, 2))
# atr_period_range             = list(range(7, 22, 1))
# atr_mult_range               = np.round(np.arange(4.0, 10.0 + 0.05, 0.2), 1).tolist()
# cooldown_bars_range          = list(range(3, 16, 1))
# min_profit_pct_range         = np.round(np.arange(0.003, 0.025 + 0.0005, 0.002), 3).tolist()

# print("=" * 85)
# print("DYNAMIC RANGE RUANG PARAMETER TERBENTUK:")
# print(f"  * macro_spread_threshold : {len(macro_spread_threshold_range)} nilai ({macro_spread_threshold_range[0]} s.d. {macro_spread_threshold_range[-1]})")
# print(f"  * ema_micro              : {len(ema_micro_range)} nilai ({ema_micro_range[0]} s.d. {ema_micro_range[-1]})")
# print(f"  * atr_period             : {len(atr_period_range)} nilai ({atr_period_range[0]} s.d. {atr_period_range[-1]})")
# print(f"  * atr_mult               : {len(atr_mult_range)} nilai ({atr_mult_range[0]} s.d. {atr_mult_range[-1]})")
# print(f"  * cooldown_bars          : {len(cooldown_bars_range)} nilai ({cooldown_bars_range[0]} s.d. {cooldown_bars_range[-1]})")
# print(f"  * min_profit_pct         : {len(min_profit_pct_range)} nilai ({min_profit_pct_range[0]} s.d. {min_profit_pct_range[-1]})")

# # -----------------------------------------------------------------------------
# # 3. KALKULASI & RANDOMIZED SAMPLING (MONTE CARLO SAMPLING)
# # -----------------------------------------------------------------------------
# all_combinations = list(itertools.product(
#     macro_spread_threshold_range,
#     ema_micro_range,
#     atr_period_range,
#     atr_mult_range,
#     cooldown_bars_range,
#     min_profit_pct_range
# ))
# total_combinations = len(all_combinations)
# MAX_SAMPLES = 15000

# if total_combinations > MAX_SAMPLES:
#     random.seed(42)  # Menjamin reprodusibilitas sampling Monte Carlo
#     active_combinations = random.sample(all_combinations, MAX_SAMPLES)
#     print(f"\nTotal Kombinasi Teoretis : {total_combinations:,}")
#     print(f"Sampling Acak Diambil   : {len(active_combinations):,} kombinasi (Batas Maksimal Aman PC)")
# else:
#     active_combinations = all_combinations
#     print(f"\nTotal Kombinasi         : {total_combinations:,}")

# TARGET_INSTRUMENTS = ['SPY', 'QQQ', 'DJI']
# OUTPUT_EXCEL_RANDOM = BASE_DIR / "Dynamic_Random_Search_Optimization.xlsx"
# CPU_CORES = os.cpu_count() or 4
# print(f"Instrumen Target        : {TARGET_INSTRUMENTS} (Long-Only)")
# print(f"Alokasi Paralel CPU     : {CPU_CORES} Cores")
# print(f"Target Berkas Luaran    : {OUTPUT_EXCEL_RANDOM}")
# print("=" * 85)

# # -----------------------------------------------------------------------------
# # 4. PRE-LOADING DATA (ZERO DISK I/O OVERHEAD PADA PEKERJA PARALEL)
# # -----------------------------------------------------------------------------
# print("\nMemuat basis data intra-trade untuk pekerja paralel...")
# preloaded_data = {}
# for asset in TARGET_INSTRUMENTS:
#     df_raw = preprocess_asset_data(asset)
#     paths = get_asset_file_paths(asset)
#     d1 = load_clean_data(paths['1d'])
#     d1['ema_fast'] = pd.Series(calc_ema_series(d1['close'], 21), index=d1.index).shift(1)
#     d1['ema_slow'] = pd.Series(calc_ema_series(d1['close'], 63), index=d1.index).shift(1)
#     d1_sub = d1[['date', 'ema_fast', 'ema_slow']]
#     df_merged = pd.merge(df_raw.reset_index(), d1_sub, on='date', how='left')
#     df_merged.ffill(inplace=True)
#     df_merged.set_index('datetime', inplace=True)
#     preloaded_data[asset] = df_merged

# print("Basis data berhasil dipersiapkan. Memulai eksekusi paralel...")

# # -----------------------------------------------------------------------------
# # 5. EKSEKUSI PARALEL MULTIPROCESSING DENGAN TQDM PROGRESS BAR
# # -----------------------------------------------------------------------------
# results_list = []
# start_optim_time = time.time()

# with concurrent.futures.ProcessPoolExecutor(
#     max_workers=CPU_CORES,
#     initializer=optimization_worker.init_worker,
#     initargs=(preloaded_data,)
# ) as executor:
#     futures = [executor.submit(optimization_worker.evaluate_combination_task, combo) for combo in active_combinations]
#     for fut in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Randomized Search", unit="combo"):
#         try:
#             results_list.append(fut.result())
#         except Exception as e:
#             pass

# total_time = time.time() - start_optim_time
# print(f"\nIterasi selesai dalam {total_time/60:.2f} menit ({len(results_list)/total_time:.1f} combo/detik)")

# # -----------------------------------------------------------------------------
# # 6. PEMERINGKATAN & EKSPOR KE EXCEL (Dynamic_Random_Search_Optimization.xlsx)
# # -----------------------------------------------------------------------------
# df_optim = pd.DataFrame(results_list)

# # Urutkan secara menurun (Descending) berdasarkan Avg_Total_Return
# df_optim.sort_values(by=['Avg_Total_Return', 'Avg_Max_Drawdown'], ascending=[False, False], inplace=True)
# df_optim.reset_index(drop=True, inplace=True)

# # Urutan kolom terstandarisasi institusional
# final_columns = [
#     'Avg_Total_Return', 'Avg_Max_Drawdown',
#     'SPY_Return (%)', 'QQQ_Return (%)', 'DJI_Return (%)',
#     'SPY_MaxDD (%)', 'QQQ_MaxDD (%)', 'DJI_MaxDD (%)',
#     'macro_spread_threshold', 'ema_micro', 'atr_period', 'atr_mult',
#     'cooldown_bars', 'min_profit_pct',
#     'SPY_Trades', 'QQQ_Trades', 'DJI_Trades'
# ]
# df_optim_final = df_optim[final_columns]

# # Ekspor ke Excel dengan Styling Profesional
# with pd.ExcelWriter(OUTPUT_EXCEL_RANDOM, engine='openpyxl') as writer:
#     df_optim_final.to_excel(writer, sheet_name='Ranked_Universal_Parameters', index=False)
#     df_optim_final.head(100).to_excel(writer, sheet_name='Top_100_Universal', index=False)

# wb = openpyxl.load_workbook(OUTPUT_EXCEL_RANDOM)
# header_font = Font(name='Segoe UI', size=11, bold=True, color='FFFFFF')
# header_fill = PatternFill(start_color='1F4E78', end_color='1F4E78', fill_type='solid')

# for ws in wb.worksheets:
#     ws.views.sheetView[0].showGridLines = True
#     for cell in ws[1]:
#         cell.font = header_font
#         cell.fill = header_fill
#         cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    
#     for col in ws.columns:
#         col_letter = get_column_letter(col[0].column)
#         max_len = max(len(str(c.value or '')) for c in col)
#         ws.column_dimensions[col_letter].width = max(max_len + 3, 13)

# wb.save(OUTPUT_EXCEL_RANDOM)

# print("=" * 85)
# print(f"BERKAS HASIL OPTIMASI TERSIMPAN: {OUTPUT_EXCEL_RANDOM}")
# print("=" * 85)
# print("\n>>> TOP 10 PARAMETER UNIVERSAL TERBAIK (RATA-RATA TOTAL RETURN TERTINGGI):")
# display(df_optim_final.head(10))
